# 4. Supervised vs unsupervised

The two models see the **same clips** and are scored on the **same game-held-out folds**.

* **Supervised:** LightGBM trained on labeled pitch types, using pose features (and, in the fused version, the stats model's out-of-fold probabilities as a prior).
* **Unsupervised:** standardize the pose features, apply PCA, then a Gaussian mixture with k chosen by BIC. **No labels are used to form clusters.** Labels come in only afterwards, to (a) measure how well the clusters match pitch types (purity, ARI, NMI) and (b) map each cluster to a pitch distribution on the training folds so it can be scored like the supervised model.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))          # so cache/ and reports/ live at the repo root
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pitchscout.workspace import Config, get_pitches, get_clips
from pitchscout import plotting as P

# DEMO=True runs on synthetic data with no network (numbers are NOT real).
DEMO = os.environ.get("PITCHSCOUT_DEMO") == "1"
cfg = Config(pitcher="Mason Miller", start="2024-03-28", end="2025-09-28", n_clips=600, demo=DEMO)
if DEMO:
    print("DEMO MODE: synthetic data, results do not describe a real pitcher")

In [ ]:
from pitchscout import baseline, fusion, unsupervised
df, throws = get_pitches(cfg)
base = baseline.fit_baseline(df)
clips = get_clips(cfg, df, throws)
idx = clips.pose.index.to_numpy()
y, games, prior = df.loc[idx, "pitch_type"].to_numpy(), df.loc[idx, "game_pk"].to_numpy(), base.oof_proba[idx]
sup = fusion.fit_fusion(clips.pose, prior, y, games, base.classes)
uns = unsupervised.fit_unsupervised(clips.pose, y, games, base.classes, prior=prior)

## Head-to-head

In [ ]:
rows = {
    "Overall mix": (sup.metrics["naive_mix"], "baseline"),
    "Stats only": (sup.metrics["stats_only"], "baseline"),
    "Supervised: vision only": (sup.metrics["vision_only"], "supervised"),
    "Unsupervised: clusters only": (uns.metrics["clusters_only"], "unsupervised"),
    "Supervised: stats + vision": (sup.metrics["stats_plus_vision"], "supervised"),
    "Unsupervised: stats + clusters": (uns.metrics["stats_plus_clusters"], "unsupervised"),
}
table = pd.DataFrame({k: {**v[0], "family": v[1]} for k, v in rows.items()}).T
table[["accuracy", "log_loss"]] = table[["accuracy", "log_loss"]].astype(float)
Path = __import__("pathlib").Path; Path("reports").mkdir(exist_ok=True)
if not DEMO: table.to_csv("reports/model_comparison.csv")
table

In [ ]:
P.compare_models(table, f"{cfg.pitcher}: supervised vs unsupervised ({len(idx)} clips, game-grouped CV)"); plt.show()

## How well do the unsupervised clusters line up with pitch types?

In [ ]:
print(f"k = {uns.k} clusters (BIC)")
pd.Series(uns.alignment).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(list(uns.bic), list(uns.bic.values()), "o-", color=P.SERIES[0]); ax.axvline(uns.k, color=P.SERIES[1], ls="--")
ax.set_xlabel("k"); ax.set_ylabel("BIC (lower is better)"); ax.set_title("Choosing the number of clusters", loc="left"); plt.show()
P.heatmap(uns.composition.rename(index=lambda c: f"Cluster {c+1}"), "Pitch-type share inside each cluster"); plt.show()

## Map of deliveries (PCA), colored by true pitch vs by cluster

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True, sharey=True)
col = P.pitch_colors(base.classes)
for c in base.classes:
    m = y == c
    ax[0].scatter(*uns.embedding[m].T, s=16, color=col[c], label=P.name(c), edgecolor="white", lw=.4)
ax[0].legend(frameon=False); ax[0].set_title("True pitch type", loc="left")
for k in range(uns.k):
    m = uns.clusters == k
    ax[1].scatter(*uns.embedding[m].T, s=16, color=P.SERIES[k % 8] if k < 8 else "#898781", label=f"Cluster {k+1}", edgecolor="white", lw=.4)
ax[1].legend(frameon=False, fontsize=8); ax[1].set_title("Unsupervised cluster (no labels)", loc="left")
for a in ax: a.set_xlabel("PC 1")
ax[0].set_ylabel("PC 2"); plt.show()

## Average delivery per cluster

In [ ]:
from pitchscout.pose import JOINTS, BONES
fig, axes = plt.subplots(uns.k, 1, figsize=(8, 1.9 * uns.k))
for k, ax in enumerate(np.atleast_1d(axes)):
    top = uns.composition.loc[k].idxmax()
    P.skeleton(ax, uns.mean_pose[k], JOINTS, BONES, title=f"Cluster {k+1} (n={uns.cluster_sizes[k]}), mostly {P.name(top)}")
plt.tight_layout(); plt.show()

## What the supervised vision model looks at

Per-pitch TreeSHAP contributions from LightGBM, summed by body part and time step before release.

In [ ]:
P.hbar(fusion.part_importance(sup).head(12), "Share of the supervised model's attention"); plt.show()
grid = fusion.attention_grid(sup).head(12)
step = (cfg.window_s - cfg.cutoff_s) / fusion.N_BINS
grid.columns = [f"-{cfg.window_s - step * (b + .5):.2f}s" for b in range(fusion.N_BINS)]
P.heatmap(grid, "Where and when (mean |contribution|)", fmt="{:.2f}"); plt.show()

In [ ]:
for k, c in enumerate(sup.classes):
    g = fusion.attention_grid(sup, mask=sup.y == c, cls=k, signed=True).head(8)
    g.columns = grid.columns
    P.heatmap(g, f"Tells for {P.name(c)} (blue pushes toward it, red away)", fmt="{:+.2f}", diverging=True); plt.show()

## Save this run to the database

The same numbers go into `model_runs` / `model_results`, tagged with the git commit, so the paper's tables can be traced back to the code that made them.

In [ ]:
from pitchscout.workspace import save_run
from pitchscout.pipeline import clip_rows, stats_rows
run_id = save_run(cfg, "notebook:04", df, {"stats": stats_rows(base), "clips": clip_rows(sup, uns)},
                  n_clips=len(idx), summary={"unsupervised": {"k": uns.k, **uns.alignment}})
print("saved run", run_id)